In [138]:
# import libraries
import pandas as pd
import re
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [11]:
# load the data
with open("../01_data/annotations.json", "r") as f:
    data = json.load(f)

# convert the data into the optimal structure
optimal_data = []

for task in data:
    text = task["data"]["sentence"]
    results = task["annotations"][0]["result"]
    labels = [r["value"]["text"] for r in results]
    spans = [
        {
            "start": r["value"]["start"],
            "end": r["value"]["end"],
            "text": r["value"]["text"],
            "label": r["value"]["labels"][0]
        }
        for r in results if r["type"] == "labels"
    ]
    optimal_data.append({"sentence": text,
                         "annotations": spans})

In [108]:
def augmentation_non_entity(task):

    # extract text and annotations
    text = task["sentence"]
    entities = task["annotations"]

    # augment the whole text and return if no labels present
    if not entities:
        new_sentence = aug.augment(text)[0]
        new_task = {
            "sentence": new_sentence,
            "annotations": []
        }
        return new_task
    
    # variables to store the new sentence and keep track of new indices
    new_sentence = ""
    span_indices = []
    prev_end = 0

    # loop through annotations
    for ent in entities:

        # get start, end, text and label of the span
        start, end = ent["start"], ent["end"]
        ent_text = ent["text"]
        ent_label = ent["label"]

        # extract prefix and count number of leading and trailing whitespaces
        prefix = text[prev_end:start]
        leading_wspaces = len(prefix) - len(prefix.lstrip(" "))
        trailing_wspaces = len(prefix) - len(prefix.rstrip(" "))

        # augment the core prefix and add the correct number of whitespaces
        core_prefix = prefix.strip(" ")
        if core_prefix:
            core_prefix = aug.augment(core_prefix)[0]
        augmented_prefix = ' ' * leading_wspaces + core_prefix + ' ' * trailing_wspaces

        # get the new span indices
        new_start = len(new_sentence) + len(augmented_prefix)
        new_end = new_start + len(ent_text)

        # append the annotations data to the list
        span_indices.append({
            "start": new_start,
            "end": new_end,
            "text": ent_text,
            "label": ent_label
        })

        # update the new sentence and index variable
        new_sentence += augmented_prefix + ent_text
        prev_end = end

    # add rest of the sentence after the last annotation 
    suffix = text[prev_end:]
    leading_spaces = len(suffix) - len(suffix.lstrip(' '))
    trailing_spaces = len(suffix) - len(suffix.rstrip(' '))
    core_suffix = suffix.strip(' ')
    if core_suffix:
        core_suffix = aug.augment(core_suffix)[0]
    augmented_suffix = ' ' * leading_spaces + core_suffix + ' ' * trailing_spaces
    new_sentence += augmented_suffix

    return {"sentence": new_sentence, "annotations": span_indices}

def augmentation_entity(task):

    # extract text and annotations
    text = task["sentence"]
    entities = task["annotations"]
    
    # augment the whole text and return if no labels present
    if not entities:
        new_sentence = aug.augment(text)[0]
        new_task = {
            "sentence": new_sentence,
            "annotations": []
        }
        return new_task
    
    # variables to store the new sentence and keep track of new indices
    new_sentence = ""
    span_indices = []
    prev_end = 0

    # loop through annotations
    for ent in entities:

        # get start, end, text and label of the span
        start, end = ent["start"], ent["end"]
        ent_text = ent["text"]
        ent_label = ent["label"]

        # extract prefix
        prefix = text[prev_end:start]

        # augment the annotation
        augmented_entity = aug.augment(ent_text)[0]

        # get the new span indices
        new_start = len(new_sentence) + len(prefix)
        new_end = new_start + len(augmented_entity)

        # append the annotations data to the list
        span_indices.append({
            "start": new_start,
            "end": new_end,
            "text": augmented_entity,
            "label": ent_label
        })

        # update the new sentence and index variable
        new_sentence += prefix + augmented_entity
        prev_end = end

    # add rest of the sentence after the last annotation 
    suffix = text[prev_end:]
    new_sentence += suffix

    return {"sentence": new_sentence, "annotations": span_indices}

In [121]:
import nlpaug.augmenter.word as naw

aug = naw.SynonymAug(aug_src='wordnet', aug_p = 0.5)

augmentation_entity(optimal_data[9])

{'sentence': 'May I point out to the hon. Lady that, in fact, global positioning system are often very aware of the services that are needed?',
 'annotations': [{'start': 48,
   'end': 73,
   'text': 'global positioning system',
   'label': 'sg_pos'}]}

In [ ]:
# load the model
checkpoint = "HuggingFaceTB/SmolLM-1.7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model = AutoModelForCausalLM.from_pretrained(checkpoint).to(device)

In [135]:
def compile_augmentation_prompt(task, tokenizer):
    sentence = task["sentence"]
    entities = task["annotations"]

    entity_texts = [e["text"] for e in entities]
    entity_list = ", ".join(entity_texts)

    # Build chat template
    if not entities:
        chat = [
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant that paraphrases sentences. "
                    "Paraphrase the following sentence in one single sentence while maintaining its meaning."
                ),
            },
            {
                "role": "user",
                "content": f"Sentence: {sentence}",
            },
        ]
    else:
        chat = [
            {
                "role": "system",
                "content": (
                    f"You are a helpful assistant that paraphrases sentences. "
                    f"Paraphrase the following sentence in one single sentence while keeping these entities unchanged: {entity_list}. "
                    "Use each of these entities exactly as often as they appear in this list."
                ),
            },
            {
                "role": "user",
                "content": f"Sentence: {sentence}",
            },
        ]

    prompt = tokenizer.apply_chat_template(
        chat,
        tokenize=False,
        add_generation_prompt=True,
    )

    return prompt

In [141]:
def find_entity_span(task, augmented_sentence):
    sentence = task["sentence"]
    entities = task["annotations"]

    if not entities:
        new_task = {
            "sentence": augmented_sentence,
            "annotations": []
        }
        return new_task

    new_entities = []
    search_start = 0

    for ent in entities:
        ent_text = ent["text"]
        label = ent["label"]

        match = re.search(re.escape(ent_text), augmented_sentence[search_start:], re.IGNORECASE)

        if match:
            start = search_start + match.start()
            end = search_start + match.end()
            new_entities.append({
                "start": start,
                "end": end,
                "text": augmented_sentence[start:end],
                "label": label
            })
            search_start = end

        # only proceed if all entities could have been found
        else:
            return None
    
    new_task = {
        "sentence": augmented_sentence,
        "annotations": new_entities
    }

    return new_task

In [146]:
# test the augmentation
task = optimal_data[13]
prompt = compile_augmentation_prompt(task, tokenizer=tokenizer)
prompt_ids = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)
outputs = model.generate(**prompt_ids)
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
answer_text = generated_text.split("assistant\n")[-1]
new_task = find_entity_span(task, answer_text)
new_task

{'sentence': 'A special subject for me is online violence, abuse, and bullying, particularly against women and girls.',
 'annotations': [{'start': 87, 'end': 92, 'text': 'women', 'label': 'sg_pos'},
  {'start': 97, 'end': 102, 'text': 'girls', 'label': 'sg_pos'}]}